# 04_test.ipynb — Final Test Set Evaluation under FREEZE Gate
This notebook evaluates the models on the official test set:
- Enforces `experiments/FREEZE.md` barrier.
- Evaluates the models selected by validation from `final_choice.json` once per dataset.
- Evaluates all other finished runs once, labeled: `"reported, not used for selection"`.
- Refuses to overwrite any existing `test_results.json`.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

if not os.path.exists("eval_test.py"):
    if os.path.exists("flower-lite"):
        os.chdir("flower-lite")
    elif os.path.exists("/kaggle/working/flower-lite"):
        os.chdir("/kaggle/working/flower-lite")

print("Working Directory:", os.getcwd())
assert os.path.exists("eval_test.py"), "eval_test.py must exist!"


In [ ]:
# Enforce FREEZE gate
os.makedirs("experiments", exist_ok=True)
freeze_file = Path("experiments/FREEZE.md")
with open(freeze_file, "w") as f:
    f.write("# FREEZE\nModel selection finalized. Test set unlocked for single-pass evaluation.\n")
print("FREEZE.md written. Test evaluation unlocked.")

choice_file = Path("/kaggle/working/final_choice.json")
if not choice_file.exists():
    choice_file = Path("experiments/final_choice.json")

if not choice_file.exists():
    raise FileNotFoundError("final_choice.json not found! Run 03_main.ipynb first.")

with open(choice_file, "r") as f:
    final_choice = json.load(f)

print("Final choice loaded:")
print(json.dumps(final_choice, indent=2))


In [ ]:
# Evaluate chosen models on official test set
test_outputs = {}

for dataset, info in final_choice.items():
    run_id = info["run_id"]
    weight_type = info.get("weight_type", "ema")
    
    print(f"\nEvaluating WINNER for {dataset.upper()} (Run ID: {run_id}, Weights: {weight_type.upper()})")
    
    ckpt_dir = f"/kaggle/working/runs/{run_id}"
    if not os.path.exists(ckpt_dir):
        ckpt_dir = f"experiments/{run_id}"
        
    cmd = [
        sys.executable, "eval_test.py",
        "--run-id", run_id,
        "--dataset", dataset,
        "--weight-type", weight_type,
        "--checkpoint-dir", ckpt_dir,
        "--output-dir", ckpt_dir,
        "--data-root", "/kaggle/working/data"
    ]
    
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print("STDERR:\n", res.stderr)
        raise RuntimeError(f"eval_test.py failed for {run_id}")
        
    res_json_path = os.path.join(ckpt_dir, "test_results.json")
    with open(res_json_path, "r") as f:
        test_outputs[dataset] = json.load(f)


In [ ]:
import json
from pathlib import Path

import pandas as pd

runs_base = Path("/kaggle/working/runs")
if not runs_base.exists():
    runs_base = Path("experiments")

all_evaluations = []

for dataset, res in test_outputs.items():
    all_evaluations.append({
        "dataset": dataset,
        "run_id": res["run_id"],
        "weight_type": res["weight_type"],
        "top1": res["top1"],
        "top5": res["top5"],
        "macro_f1": res["macro_f1"],
        "nll": res["nll"],
        "ece": res["ece"],
        "role": "SELECTED WINNER (chosen by validation)"
    })

for run_dir in sorted(runs_base.iterdir()):
    if not run_dir.is_dir():
        continue
    run_id = run_dir.name
    if any(e["run_id"] == run_id for e in all_evaluations):
        continue
        
    dataset = "cifar10" if "c10_" in run_id or "cifar10" in run_id else "cifar100"
    res_path = run_dir / "test_results.json"
    
    if not res_path.exists():
        cmd = [
            sys.executable, "eval_test.py",
            "--run-id", run_id,
            "--dataset", dataset,
            "--weight-type", "ema",
            "--checkpoint-dir", str(run_dir),
            "--output-dir", str(run_dir),
            "--data-root", "/kaggle/working/data"
        ]
        sub_res = subprocess.run(cmd, capture_output=True, text=True)
        if sub_res.returncode != 0:
            print(f"Skipping {run_id}: evaluation returned code {sub_res.returncode}")
            continue
            
    with open(res_path, "r") as f:
        metrics = json.load(f)
        all_evaluations.append({
            "dataset": dataset,
            "run_id": run_id,
            "weight_type": metrics.get("weight_type", "ema"),
            "top1": metrics.get("top1"),
            "top5": metrics.get("top5"),
            "macro_f1": metrics.get("macro_f1"),
            "nll": metrics.get("nll"),
            "ece": metrics.get("ece"),
            "role": "reported, not used for selection"
        })

df_all = pd.DataFrame(all_evaluations)
print("\n============================== FINAL OFFICIAL TEST RESULTS ==============================")
print(df_all.to_string(index=False))

with open("/kaggle/working/all_test_evaluations.json", "w") as f:
    json.dump(all_evaluations, f, indent=4)
print("\nWrote /kaggle/working/all_test_evaluations.json")
